# Ingest RHOAI documentation into the Llama Stack vector store

This notebook follows the same pattern as the AG News ingestion notebook, but loads
prepared chunks from S3 (output of the Docling data preparation notebook or the KFP
pipeline) and ingests them into the Llama Stack vector store with rich product metadata.
Each chunk becomes a file uploaded through the Llama Stack Files API and attached to
the vector store with document-level attributes for downstream filtering and retrieval.

> **Dev vs Production:** This notebook writes to a `-dev` vector store so
> development/testing never overwrites production data created by the KFP pipeline.

## 1. Connect to Llama Stack and verify models

Verify the runtime is ready and confirm the embedding model is available.

In [ ]:
import os
from llama_stack_client import LlamaStackClient

base_url = os.environ.get("LLAMA_STACK_BASE_URL",
    "http://lsd-enterprise-rag-service.enterprise-rag.svc.cluster.local:8321")
embedding_model = os.environ.get("RHOAI_STAGE230_EMBEDDING_MODEL",
    "sentence-transformers/nomic-ai/nomic-embed-text-v1.5")

client = LlamaStackClient(base_url=base_url)

models = client.models.list()
model_list = list(models) if not isinstance(models, list) else models
print(f"Llama Stack endpoint: {base_url}")
print(f"Embedding model: {embedding_model}\n")
print("Available models:")
for m in model_list:
    model_id = m.id if hasattr(m, 'id') else m.get('id', '?')
    model_type = getattr(m, 'model_type', m.get('type', '?'))
    print(f"  {model_id} ({model_type})")

## 2. Load prepared chunks from S3

Load the JSONL output from the Docling data preparation notebook or the KFP pipeline.

In [ ]:
import json
import warnings

warnings.filterwarnings("ignore", message=".*Unverified HTTPS.*")
warnings.filterwarnings("ignore", message=".*InsecureRequestWarning.*")

import boto3
from botocore.config import Config

s3 = boto3.client(
    "s3",
    endpoint_url=os.environ["AWS_S3_ENDPOINT"],
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    region_name=os.environ.get("AWS_DEFAULT_REGION", "us-east-1"),
    verify=False,
    config=Config(signature_version="s3v4"),
)

s3_bucket = os.environ["AWS_S3_BUCKET"]
chunks_key = "processed/rhoai-product-docs/rhoai-3.4-product-docs-docling-notebook-chunks.jsonl"

body = s3.get_object(Bucket=s3_bucket, Key=chunks_key)["Body"].read().decode("utf-8")
records = [json.loads(line) for line in body.splitlines() if line.strip()]

print(f"Loaded {len(records)} chunks from s3://{s3_bucket}/{chunks_key}\n")

guides = {}
for r in records:
    slug = r.get("guide_slug", "unknown")
    guides[slug] = guides.get(slug, 0) + 1

print("Chunks per guide:")
for slug, count in sorted(guides.items()):
    print(f"  {slug}: {count}")

## 3. Create the vector store with metadata

Create a new vector store configured with the embedding model and corpus-level metadata
(tenant_id, version, product, etc.). This mirrors the AG News ingestion pattern.

In [ ]:
import inspect

VECTOR_STORE_NAME = "stage230-rhoai-34-product-docs-dev"
VECTOR_PROVIDER = "pgvector"

def find_vector_store(client, name):
    stores = client.vector_stores.list()
    store_list = list(stores) if not isinstance(stores, list) else stores
    for store in store_list:
        store_id = store.id if hasattr(store, 'id') else store.get('id')
        store_name = store.name if hasattr(store, 'name') else store.get('name')
        if store_name == name:
            return store_id
    return None

existing_id = find_vector_store(client, VECTOR_STORE_NAME)
if existing_id:
    print(f"Deleting existing vector store: {existing_id}")
    client.vector_stores.delete(vector_store_id=existing_id)

first_record = records[0]
store_metadata = {
    "tenant_id": first_record["tenant_id"],
    "version_no": first_record["version_no"],
    "corpus": first_record["corpus"],
    "environment": "demo",
    "language": first_record["language"],
    "product": first_record["product"],
    "product_version": first_record["product_version"],
    "provider_id": VECTOR_PROVIDER,
    "embedding_model": embedding_model,
}

create_kwargs = {
    "name": VECTOR_STORE_NAME,
    "metadata": store_metadata,
    "extra_body": {"provider_id": VECTOR_PROVIDER},
}
if "embedding_model" in inspect.signature(client.vector_stores.create).parameters:
    create_kwargs["embedding_model"] = embedding_model
else:
    create_kwargs["extra_body"]["embedding_model"] = embedding_model

store = client.vector_stores.create(**create_kwargs)
vector_store_id = store.id if hasattr(store, 'id') else store.get('id', store.get('vector_store_id'))

print(f"Created vector store: {VECTOR_STORE_NAME}")
print(f"  ID: {vector_store_id}")
print(f"  Provider: {VECTOR_PROVIDER}")
print(f"  Embedding: {embedding_model}")
print(f"\nVector store metadata:")
for k, v in store_metadata.items():
    print(f"  {k}: {v}")

## 4. Upload files through the Files API

Create one file per chunk and upload it through the Llama Stack Files API. This mirrors
the AG News pattern where each article becomes a file.

In [ ]:
import tempfile
from pathlib import Path

ALLOWED_ATTRIBUTES = {
    "id", "access_tier", "chunk_index", "corpus", "document_title",
    "document_type", "documentation_category", "guide_slug", "language",
    "matched_terms", "page_start", "page_end", "preparation_method",
    "product", "product_version", "retrieved_url", "source",
    "source_authority", "source_file", "source_format", "source_url",
    "tenant_id", "topic", "version", "version_no",
}

def record_attributes(record):
    attrs = {k: record[k] for k in sorted(ALLOWED_ATTRIBUTES) if k in record and record[k] is not None}
    if isinstance(attrs.get("matched_terms"), list):
        attrs["matched_terms"] = ",".join(str(t) for t in attrs["matched_terms"])
    attrs["record_id"] = record["id"]
    return attrs

uploaded = 0
with tempfile.TemporaryDirectory(prefix="rhoai-docs-ingest-") as tmpdir:
    tmp = Path(tmpdir)
    for i, record in enumerate(records):
        chunk_path = tmp / f"{record['id']}.txt"
        chunk_path.write_text(
            f"{record['title']}\n"
            f"Source: {record['source_url']}\n"
            f"Topic: {record['topic']}\n\n"
            f"{record['text']}\n",
            encoding="utf-8",
        )
        with chunk_path.open("rb") as f:
            file_obj = client.files.create(file=f, purpose="assistants")

        file_id = file_obj.id if hasattr(file_obj, 'id') else file_obj.get('id', file_obj.get('file_id'))

        client.vector_stores.files.create(
            vector_store_id=vector_store_id,
            file_id=file_id,
            attributes=record_attributes(record),
        )
        uploaded += 1
        if uploaded % 25 == 0 or uploaded == len(records):
            print(f"  Uploaded {uploaded}/{len(records)} chunks...")

print(f"\nIngestion complete: {uploaded} chunks uploaded to vector store '{VECTOR_STORE_NAME}'")

## 5. Verify ingestion

Check the vector store file counts to confirm all files were processed.

In [ ]:
store_info = client.vector_stores.retrieve(vector_store_id=vector_store_id)
file_counts = store_info.file_counts if hasattr(store_info, 'file_counts') else store_info.get('file_counts', {})

completed = int(getattr(file_counts, 'completed', 0) or (file_counts.get('completed', 0) if isinstance(file_counts, dict) else 0))
total = int(getattr(file_counts, 'total', 0) or (file_counts.get('total', 0) if isinstance(file_counts, dict) else 0))
failed = int(getattr(file_counts, 'failed', 0) or (file_counts.get('failed', 0) if isinstance(file_counts, dict) else 0))
in_progress = int(getattr(file_counts, 'in_progress', 0) or (file_counts.get('in_progress', 0) if isinstance(file_counts, dict) else 0))

print(f"Vector store: {VECTOR_STORE_NAME} ({vector_store_id})")
print(f"  Completed:   {completed}")
print(f"  In progress: {in_progress}")
print(f"  Failed:      {failed}")
print(f"  Total:       {total}")

if failed > 0 or in_progress > 0:
    print("\n\u26a0 Some files are still processing or have failed")
else:
    print(f"\n\u2713 All {completed} files ingested successfully")

## 6. Quick search test

Run a simple search to verify the ingested content is queryable.

In [ ]:
test_query = "How does OpenShift AI use Llama Stack for RAG?"

results = client.vector_stores.search(
    vector_store_id=vector_store_id,
    query=test_query,
    max_num_results=3,
)

result_list = list(results) if not isinstance(results, list) else results
if hasattr(results, 'data'):
    result_list = list(results.data)

print(f"Search query: '{test_query}'")
print(f"Results: {len(result_list)}\n")

for i, hit in enumerate(result_list):
    text = ""
    content = getattr(hit, 'content', None) or (hit.get('content') if isinstance(hit, dict) else None)
    if isinstance(content, list):
        text = "\n".join(str(getattr(p, 'text', '') or p.get('text', '')) for p in content if (getattr(p, 'text', None) or (p.get('text') if isinstance(p, dict) else None)))
    elif isinstance(content, str):
        text = content
    else:
        text = str(getattr(hit, 'text', '') or (hit.get('text', '') if isinstance(hit, dict) else ''))

    score = getattr(hit, 'score', None) or (hit.get('score') if isinstance(hit, dict) else None)
    attrs = getattr(hit, 'attributes', None) or (hit.get('attributes') if isinstance(hit, dict) else {})
    guide = (attrs.get('guide_slug', '?') if isinstance(attrs, dict) else getattr(attrs, 'guide_slug', '?'))
    topic = (attrs.get('topic', '?') if isinstance(attrs, dict) else getattr(attrs, 'topic', '?'))

    print(f"[{i+1}] score={score}, guide={guide}, topic={topic}")
    print(f"    {text[:200]}...")
    print()

print("\u2713 Vector store is queryable. Open Retrieval_pipeline_rhoai_docs.ipynb for the full retrieval pipeline.")

---

This notebook mirrors the AG News ingestion pattern for RHOAI product documentation.
The prepared chunks (from the Docling data preparation notebook or the KFP pipeline)
are now indexed in the Llama Stack vector store with rich product metadata.

**Next step:** Open `Retrieval_pipeline_rhoai_docs.ipynb` to run metadata-filtered
retrieval, reranking, and grounded answer generation against this corpus.